# Gemma 4 26B-A4B (QAT Q4_0 GGUF) für Leadbrief auf Kaggle testen

Dieses Notebook lädt das **offizielle Google-Modell** `google/gemma-4-26B-A4B-it-qat-q4_0-gguf` (quantization-aware trainiertes Q4_0, ca. 13,5 GB) und betreibt es mit `llama-server` (llama.cpp, CUDA). Der Server spricht nativ das OpenAI-Chat-Completions-Format inklusive `response_format.json_schema` und API-Key. Ein **temporärer** HTTPS-Tunnel verbindet ihn mit der Leadbrief-Chrome-Erweiterung.

**Warum nicht Transformers + bitsandbytes?** Die Experten des MoE-Modells sind 3D-Parameter, die bitsandbytes nicht quantisiert; sie blieben in FP16 (> 40 GiB) und passen nicht auf zwei T4. Das GGUF-Modell hat dagegen alle Gewichte in 4 Bit und passt mit Kontext komplett in den VRAM beider GPUs (ca. 13,5 GB Gewichte + KV-Cache), ohne CPU-Offload.

## Vor dem Start in Kaggle

1. Notebook **privat** halten und unter **Settings → Accelerator → GPU T4 x2** auswählen. **Internet** aktivieren.
2. Kaggle-Sitzungen sind zeitlich begrenzt und können enden. Der Tunnel-Link ist temporär; das ist kein Produktionshosting.
3. Beim App-Test zuerst eine unkritische öffentliche Testseite verwenden. Website-Inhalte gehen vom Browser über Cloudflare zu Kaggle. Nicht mit echten Kontakten oder vertraulichen Inhalten testen, bevor Datenverarbeitung und Datenschutz geklärt sind.
4. Den vom Notebook erzeugten API-Token nicht teilen. Notebook-Ausgaben mit Endpoint/Token niemals veröffentlichen.

Die Erweiterung sendet bereits Chat-Completions mit `response_format.json_schema`; `llama-server` erzwingt das Schema per Grammatik, damit die bestehende App-Logik unverändert funktioniert.

**Hinweis zum Download:** Das Notebook lädt ca. 13,5 GB Modell sowie den llama.cpp-Quellcode, der mit dem Kaggle-CUDA-Toolkit gebaut wird (ca. 10-20 Minuten) und verlangt dafür 25 GiB freien temporären Speicher. Es sind keine Python-ML-Pakete nötig; PyTorch dient nur der GPU-Prüfung.

## 1. Kaggle Runtime und GPU prüfen

Erwartet werden zwei T4 mit je 16 GB VRAM. Das Q4_0-Modell benötigt ca. 13,5 GB, der 24K-Kontext (Sliding-Window-Attention) nur wenige GB zusätzlich; das Modell wird per Layer-Split gleichmäßig auf beide GPUs verteilt.

In [ ]:
import subprocess
import torch

print("PyTorch:", torch.__version__)
print("CUDA verfügbar:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Keine CUDA-GPU gefunden. In Kaggle GPU T4 x2 als Accelerator aktivieren und Session neu starten.")

gpu_count = torch.cuda.device_count()
total_vram_gib = 0.0
for index in range(gpu_count):
    props = torch.cuda.get_device_properties(index)
    free_bytes, total_bytes = torch.cuda.mem_get_info(index)
    total_vram_gib += total_bytes / 2**30
    print(f"GPU {index}: {props.name}; VRAM {total_bytes / 2**30:.1f} GiB; frei {free_bytes / 2**30:.1f} GiB")

if gpu_count < 2 or total_vram_gib < 28:
    raise RuntimeError(f"{gpu_count} GPU(s) mit zusammen {total_vram_gib:.1f} GiB erkannt. Für diesen Test GPU T4 x2 auswählen.")

try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True).stdout)
except (FileNotFoundError, subprocess.CalledProcessError):
    print("nvidia-smi ist nicht verfügbar; CUDA-Geräte wurden über PyTorch erkannt.")
print(f"Gesamt-VRAM: {total_vram_gib:.1f} GiB. Ein erfolgreicher Modellstart ist trotzdem nicht garantiert.")

## 2. Speicherplatz prüfen und Modell laden

Das GGUF-Modell von Google ist öffentlich (kein Hugging-Face-Token nötig). Es wird nach `/kaggle/temp` geladen, nicht in das 20-GB-Limit von `/kaggle/working`. Das Notebook nutzt nur die Textdatei; der Bild-Projektor (`mmproj`) wird nicht geladen.

In [ ]:
import os
import shutil

MODEL_REPO = "google/gemma-4-26B-A4B-it-qat-q4_0-gguf"
MODEL_FILE = "gemma-4-26B_q4_0-it.gguf"
MODEL_ALIAS = "gemma-4-26B-A4B-it"
WORK_ROOT = "/kaggle/temp/gemma4"
os.makedirs(WORK_ROOT, exist_ok=True)

free_gib = shutil.disk_usage(WORK_ROOT).free / 2**30
print(f"Scratch-Disk frei: {free_gib:.1f} GiB; Arbeitsordner: {WORK_ROOT}")
if free_gib < 25:
    raise RuntimeError("Zu wenig Scratch-Disk für Modell (ca. 13,5 GB) und llama.cpp (ca. 0,8 GB). Kein Download gestartet.")

from huggingface_hub import hf_hub_download

model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, local_dir=os.path.join(WORK_ROOT, "model"), token=os.environ.get("HF_TOKEN"))
model_size_gb = os.path.getsize(model_path) / 1e9
print(f"Modell: {model_path} ({model_size_gb:.2f} GB)")
if model_size_gb < 12:
    raise RuntimeError("Die Modelldatei ist unerwartet klein; Download unvollständig. Zelle erneut ausführen.")

## 3. llama.cpp (CUDA) bauen

Die vorkompilierten CUDA-Builds (12.8 und 13.5) laufen mit dem Kaggle-Treiber nicht. Deshalb wird llama.cpp `b12477` mit dem auf Kaggle vorhandenen CUDA-Toolkit direkt für die T4 (sm_75) gebaut; das dauert beim ersten Lauf etwa 10-20 Minuten. Die Build-Nummer ist fest gepinnt, damit das Verhalten reproduzierbar bleibt.

In [ ]:
import glob
import re
import time

LLAMA_TAG = "b12477"
LLAMA_SRC = os.path.join(WORK_ROOT, "llama.cpp-src")
LLAMA_BUILD = os.path.join(LLAMA_SRC, "build")
server_bin = os.path.join(LLAMA_BUILD, "bin", "llama-server")

# Die vorkompilierten CUDA-12.8/13.5-Builds laufen mit dem Kaggle-Treiber nicht (keine GPU erkannt bzw.
# "PTX was compiled with an unsupported toolchain"). Deshalb wird mit dem lokalen CUDA-Toolkit direkt für die T4 (sm_75) gebaut.
nvcc = shutil.which("nvcc") or next(iter(glob.glob("/usr/local/cuda*/bin/nvcc")), None)
if nvcc is None:
    raise RuntimeError("nvcc (CUDA-Toolkit) nicht gefunden; llama.cpp kann nicht gebaut werden.")
print(subprocess.run([nvcc, "--version"], capture_output=True, text=True).stdout.strip().splitlines()[-1])
build_env = {**os.environ, "PATH": os.path.dirname(nvcc) + ":" + os.environ["PATH"], "CUDACXX": nvcc}

if not os.path.exists(server_bin):
    build_log_path = "/kaggle/working/llama-build.log"
    print("Baue llama.cpp (ca. 10-20 Minuten). Log:", build_log_path)
    steps = []
    # CMake findet libcuda auf Kaggle nicht von selbst (CUDA::cuda_driver); daher Pfad explizit suchen.
    libcuda_candidates = (
        glob.glob("/usr/local/cuda*/**/stubs/libcuda.so", recursive=True)
        + glob.glob("/usr/lib/x86_64-linux-gnu/libcuda.so*")
        + glob.glob("/usr/local/nvidia/lib64/libcuda.so*")
    )
    if not libcuda_candidates:
        raise RuntimeError("libcuda nicht gefunden; GPU-Treiber/Accelerator prüfen.")
    libcuda = libcuda_candidates[0]
    print("libcuda:", libcuda)
    # Frischer Build-Ordner bei neuem libcuda-Pfad (kein alter CMake-Cache); sonst wird ein abgebrochener Build fortgesetzt.
    cache_file = os.path.join(LLAMA_BUILD, "CMakeCache.txt")
    if os.path.exists(cache_file) and libcuda not in open(cache_file).read():
        shutil.rmtree(LLAMA_BUILD, ignore_errors=True)
    stub_dir = os.path.join(WORK_ROOT, "cuda-stub")
    os.makedirs(stub_dir, exist_ok=True)
    stub_link = os.path.join(stub_dir, "libcuda.so")
    if not os.path.lexists(stub_link):
        os.symlink(libcuda, stub_link)
    if not os.path.isdir(LLAMA_SRC):
        steps.append(["git", "clone", "--depth", "1", "--branch", LLAMA_TAG, "https://github.com/ggml-org/llama.cpp", LLAMA_SRC])
    steps += [
        ["cmake", "-S", LLAMA_SRC, "-B", LLAMA_BUILD, "-DGGML_CUDA=ON", "-DCMAKE_CUDA_ARCHITECTURES=75-real", "-DCMAKE_BUILD_TYPE=Release", "-DLLAMA_CURL=OFF", "-DLLAMA_OPENSSL=OFF",
         f"-DCUDA_cuda_driver_LIBRARY={libcuda}", f"-DCUDA_CUDA_LIBRARY={libcuda}", f"-DCMAKE_LIBRARY_PATH={stub_dir}"],
        ["cmake", "--build", LLAMA_BUILD, "--target", "llama-server", "-j", str(os.cpu_count() or 4)],
    ]
    def run_logged(step, log):
        # Schreibt die Ausgabe ins Log und zeigt für den Build eine Fortschrittszeile (Prozent, Zeit, aktuelle Datei).
        started = time.time()
        process = subprocess.Popen(step, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=build_env)
        shown = False
        for line in process.stdout:
            log.write(line)
            match = re.match(r"\[\s*(\d+)%\]", line)
            if match:
                elapsed = int(time.time() - started)
                name = line.split()[-1].rsplit("/", 1)[-1][:50]
                print(f"\rBuild {match.group(1):>3}% | {elapsed // 60}:{elapsed % 60:02d} min | {name:<50}", end="", flush=True)
                shown = True
        process.wait()
        if shown:
            print()
        return process.returncode

    with open(build_log_path, "w") as build_log:
        for step in steps:
            if run_logged(step, build_log) != 0:
                build_log.flush()
                with open(build_log_path) as log_file:
                    tail = "".join(log_file.readlines()[-40:])
                raise RuntimeError(f"Build-Schritt fehlgeschlagen: {' '.join(step[:2])}\n" + tail)

llama_variant = "source-build"
llama_env = {**os.environ, "LD_LIBRARY_PATH": os.path.dirname(server_bin) + ":" + os.environ.get("LD_LIBRARY_PATH", "")}
listing = subprocess.run([server_bin, "--list-devices"], capture_output=True, text=True, env=llama_env)
device_text = listing.stdout + listing.stderr
print(device_text.strip())
if len(re.findall(r"^\s*CUDA\d+:", device_text, re.M)) < 2:
    raise RuntimeError("Der gebaute llama-server erkennt nicht beide GPUs. Accelerator GPU T4 x2 prüfen und Session neu starten.")
version_output = subprocess.run([server_bin, "--version"], capture_output=True, text=True, env=llama_env)
print((version_output.stdout + version_output.stderr).strip())

## 4. API-Server starten

`llama-server` läuft lokal auf `127.0.0.1:8001` und ist nur mit dem zufällig erzeugten Bearer-Token erreichbar (`LLAMA_API_KEY`; die Web-Oberfläche ist deaktiviert). Eingestellt sind: 24K Kontext, bis 4096 Ausgabe-Tokens, ein Slot, alle Layer auf den GPUs (Split 1:1) und Gemma-4-Thinking aus. Das Laden des Modells dauert einige Minuten.

In [ ]:
import secrets
import time
import httpx

LLAMA_PORT = 8001
MAX_CONTEXT_TOKENS = 24_576
# Optionales Kaggle-Secret CLIPPER_TOKEN (Add-ons → Secrets): gleicher Token in jeder Sitzung, sonst zufällig.
try:
    from kaggle_secrets import UserSecretsClient
    API_TOKEN = UserSecretsClient().get_secret("CLIPPER_TOKEN")
except Exception:
    API_TOKEN = ""
API_TOKEN = API_TOKEN or secrets.token_urlsafe(32)

if globals().get("server_process") is not None and server_process.poll() is None:
    server_process.terminate()
    server_process.wait(timeout=30)

server_log_path = "/kaggle/working/llama-server.log"
server_log = open(server_log_path, "w")
# Log-Zeitstempel sind relativ zum Serverstart; damit rechnet Abschnitt 8 sie in UTC um.
server_started_at = time.time()
server_process = subprocess.Popen(
    [
        server_bin,
        "-m", model_path,
        "--alias", MODEL_ALIAS,
        "--host", "127.0.0.1",
        "--port", str(LLAMA_PORT),
        "-c", str(MAX_CONTEXT_TOKENS),
        "-np", "1",
        "-ngl", "all",
        "-sm", "layer",
        "-ts", "1,1",
        "--jinja",
        "--reasoning", "off",
        "--chat-template-kwargs", '{"enable_thinking": false}',
        # Analyse-JSON (Systeme, 4–5 Ideen, Kontakte) braucht deutlich mehr als 1024 Tokens.
        "-n", "4096",
        "--no-webui",
        # Ab b12477 erscheinen Ladeinfos wie "offloaded N/M layers to GPU" erst ab Verbosity 4 (trace).
        "-lv", "4",
        "--log-prefix", "--log-timestamps", "--log-colors", "off",
    ],
    env={**llama_env, "LLAMA_API_KEY": API_TOKEN},
    stdout=server_log,
    stderr=subprocess.STDOUT,
)

deadline = time.time() + 900
while time.time() < deadline:
    if server_process.poll() is not None:
        break
    try:
        if httpx.get(f"http://127.0.0.1:{LLAMA_PORT}/health", timeout=5).status_code == 200:
            break
    except httpx.HTTPError:
        pass
    time.sleep(3)
else:
    server_process.terminate()
    raise RuntimeError("llama-server wurde nicht rechtzeitig bereit. Log: " + server_log_path)

if server_process.poll() is not None:
    with open(server_log_path) as log_file:
        tail = "".join(log_file.readlines()[-40:])
    raise RuntimeError("llama-server wurde beendet:\n" + tail)

print("llama-server bereit:", httpx.get(f"http://127.0.0.1:{LLAMA_PORT}/health", timeout=5).json())
with open(server_log_path) as log_file:
    server_log_text = log_file.read()
for line in server_log_text.splitlines():
    if any(key in line for key in ("offloaded", "CUDA0", "CUDA1", "Vulkan0", "Vulkan1", "KV buffer", "n_ctx")):
        print(line.rstrip())
offload = re.search(r"offloaded (\d+)/(\d+) layers to GPU", server_log_text)
# Zweiter Beleg unabhängig vom Log-Format: belegter VRAM je GPU (Layer-Split 1:1 legt auf beide GPUs mehrere GiB).
try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], capture_output=True, text=True, check=True).stdout
    gpu_used_mib = [int(value) for value in smi.split()]
except (FileNotFoundError, subprocess.CalledProcessError, ValueError):
    gpu_used_mib = []
print("Belegter VRAM je GPU (MiB):", gpu_used_mib or "unbekannt")
if offload:
    on_gpu = int(offload.group(1)) > 0
else:
    on_gpu = len(gpu_used_mib) >= 2 and min(gpu_used_mib) > 2048
if not on_gpu:
    server_process.terminate()
    tail = "".join(server_log_text.splitlines(keepends=True)[-40:])
    raise RuntimeError("llama-server nutzt die GPUs nicht (läuft auf der CPU und wäre sehr langsam). Log: " + server_log_path + "\n" + tail)
if offload:
    print(f"GPU-Offload OK: {offload.group(1)}/{offload.group(2)} Layer auf den GPUs ({llama_variant}).")
else:
    print(f"GPU-Offload OK laut nvidia-smi ({llama_variant}).")


## 5. Temporären HTTPS-Tunnel für die Erweiterung öffnen

Kaggle-Notebooks sind nicht direkt als öffentliche Webserver ausgelegt. Für einen kurzen Integrationstest startet dieser optionale Schritt einen temporären Cloudflare Quick Tunnel auf den lokalen Server. Der Link ist öffentlich erreichbar, Anfragen ohne den Bearer-Token werden aber vom Server abgewiesen. Notebook privat halten und die Ausgabe nicht teilen.

In [ ]:
import queue
import re
import threading
import urllib.request

cloudflared_path = "/kaggle/working/cloudflared"
if not os.path.exists(cloudflared_path):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        cloudflared_path,
    )
    os.chmod(cloudflared_path, 0o755)

cloudflared = subprocess.Popen(
    [cloudflared_path, "tunnel", "--url", f"http://127.0.0.1:{LLAMA_PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
cloudflared_lines = queue.Queue()
cloudflared_log_path = "/kaggle/working/cloudflared.log"
cloudflared_log = open(cloudflared_log_path, "a", buffering=1)


def _capture_cloudflared():
    for line in cloudflared.stdout:
        cloudflared_log.write(line)
        cloudflared_lines.put(line.rstrip())


threading.Thread(target=_capture_cloudflared, daemon=True).start()

tunnel_url = None
deadline = time.time() + 120
while time.time() < deadline and cloudflared.poll() is None:
    try:
        line = cloudflared_lines.get(timeout=1)
    except queue.Empty:
        continue
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break

if not tunnel_url:
    cloudflared.terminate()
    raise RuntimeError("Kein Quick-Tunnel-Link erkannt. Prüfe Kaggle-Internet und Tunnel-Logs, dann Zelle erneut ausführen.")

print("Leadbrief Endpoint:", f"{tunnel_url}/v1/chat/completions")
print("Leadbrief API-Key (Bearer):", API_TOKEN)
print("\nClipper-Orchestrator – lokal im Terminal einfügen:\n")
print(f"export CLIPPER_LLM_URL={tunnel_url}")
print(f"export CLIPPER_LLM_TOKEN={API_TOKEN}")
print("\nWICHTIG: Notebook privat halten; diesen Output weder veröffentlichen noch teilen.")

# Neue Quick-Tunnel-Namen sind oft erst nach einigen Sekunden per DNS auflösbar.
tunnel_check = None
for attempt in range(1, 19):
    try:
        tunnel_check = httpx.get(f"{tunnel_url}/health", timeout=15)
        break
    except httpx.HTTPError as error:
        print(f"Tunnel noch nicht erreichbar (Versuch {attempt}/18): {type(error).__name__}")
        time.sleep(5)
if tunnel_check is not None:
    print("Tunnel-Healthcheck:", tunnel_check.status_code, tunnel_check.text)
else:
    print("Hinweis: Der Tunnel ist aus Kaggle selbst nicht auflösbar. Das ist für die Erweiterung unkritisch; "
          "Endpoint im Browser testen (Adresse + /health). Die Modelltests unten laufen direkt auf dem lokalen Server.")


## 6. End-to-End Tests mit Leadbrief-artigen App-Prompts

Die folgenden zwei Proben gehen direkt an den lokalen Server (unabhängig vom Tunnel) und verwenden dieselbe Chat-Completions- und JSON-Schema-Form wie Leadbrief: eine Firmenanalyse und ein Mailentwurf. Sie sind Smoke-Tests, keine Qualitätsgarantie. Danach den Tunnel-Endpoint im Add-on eintragen und zunächst manuell genau eine öffentliche Testseite analysieren.

In [ ]:
import json
from types import SimpleNamespace


class LeadbriefTestClient:
    """Kleiner OpenAI-ähnlicher Testclient für den Kaggle-Endpunkt."""
    def __init__(self, base_url, api_key, timeout=600):
        self.base_url = base_url.rstrip("/")
        self.api_key = api_key
        self.timeout = timeout
        self.chat = SimpleNamespace(completions=SimpleNamespace(create=self._create))

    def _create(self, messages, response_format=None, max_tokens=None, temperature=None, **_kwargs):
        payload = {"model": MODEL_ALIAS, "messages": messages}
        if response_format is not None:
            payload["response_format"] = response_format
        if max_tokens is not None:
            payload["max_tokens"] = max_tokens
        if temperature is not None:
            payload["temperature"] = temperature
        response = httpx.post(
            f"{self.base_url}/v1/chat/completions",
            headers={"Authorization": f"Bearer {self.api_key}"},
            json=payload,
            timeout=self.timeout,
        )
        if response.status_code != 200:
            raise RuntimeError(f"HTTP {response.status_code}: {response.text[:500]}")
        data = response.json()
        return SimpleNamespace(
            choices=[SimpleNamespace(
                message=SimpleNamespace(**data["choices"][0]["message"]),
                finish_reason=data["choices"][0].get("finish_reason"),
            )],
            usage=SimpleNamespace(**data.get("usage", {})),
            model=data.get("model"),
        )


BASE_URL = f"http://127.0.0.1:{LLAMA_PORT}"
client = LeadbriefTestClient(BASE_URL, API_TOKEN, timeout=600)

analysis_schema = {
    "type": "object",
    "properties": {
        "companySummary": {"type": "string"},
        "services": {"type": "string"},
        "needs": {"type": "string"},
        "recipientEmail": {"type": "string"},
        "recipientReason": {"type": "string"},
    },
    "required": ["companySummary", "services", "needs", "recipientEmail", "recipientReason"],
    "additionalProperties": False,
}
analysis_prompt = "Analysiere diese fiktive deutsche Maschinenbau-Firma für einen überprüfbaren Leadbrief-Test. Kontext: Die Firma fertigt kundenspezifische Pumpen für Lebensmittelhersteller. Die Website nennt Wartung, Ersatzteile und ein Kontaktformular, aber keine internen IT-Systeme und keine E-Mail-Adresse. Erfinde keine Adresse; wenn keine belegt ist, recipientEmail leer lassen. Antworte nur gemäß Schema."

analysis_response = client.chat.completions.create(
    messages=[{"role": "user", "content": analysis_prompt}],
    response_format={"type": "json_schema", "json_schema": {"name": "leadbrief_analysis_test", "strict": True, "schema": analysis_schema}},
    max_tokens=700,
    temperature=0,
)
analysis_json = json.loads(analysis_response.choices[0].message.content)
assert set(analysis_json) == set(analysis_schema["required"])
assert analysis_json["recipientEmail"] == ""
print("Analyse-Smoke-Test OK:", json.dumps(analysis_json, ensure_ascii=False, indent=2))
print("Usage:", analysis_response.usage)

In [ ]:
draft_schema = {
    "type": "object",
    "properties": {
        "subject": {"type": "string"},
        "opening": {"type": "string"},
        "bulletLead": {"type": "string"},
        "bullets": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["subject", "opening", "bulletLead", "bullets"],
    "additionalProperties": False,
}
draft_prompt = "Schreibe einen kurzen, höflichen deutschen Mailentwurf für ein erstes Gespräch mit einer deutschen Pumpenmanufaktur. Nutze nur diese belegten Angaben: kundenspezifische Pumpen für Lebensmittelhersteller, Wartung und Ersatzteile. Erfinde keine Systeme, Kennzahlen oder konkreten Probleme. Gib genau drei kurze Stichpunkte."
draft_response = client.chat.completions.create(
    messages=[{"role": "user", "content": draft_prompt}],
    response_format={"type": "json_schema", "json_schema": {"name": "leadbrief_draft_test", "strict": True, "schema": draft_schema}},
    max_tokens=700,
    temperature=0,
)
draft_json = json.loads(draft_response.choices[0].message.content)
assert set(draft_json) == set(draft_schema["required"])
assert len(draft_json["bullets"]) == 3
print("Mail-Smoke-Test OK:", json.dumps(draft_json, ensure_ascii=False, indent=2))
print("Usage:", draft_response.usage)

print("\nAdd-on-Test: Chrome-Seitenleiste öffnen → Einstellungen → LLM-Endpoint und Bearer-Token aus der Tunnel-Zelle eintragen → genau eine öffentliche Testseite analysieren. Der Endpoint muss mit /v1/chat/completions enden.")

## 7. Latenz, Speicherverbrauch und Fehlerszenarien protokollieren

Der Server begrenzt den Kontext auf 24.576 Tokens; zu lange Prompts werden mit einem HTTP-Fehler abgewiesen. Es gibt absichtlich keinen automatischen kostenpflichtigen oder Cloud-Modell-Fallback. Der folgende Probe-Request protokolliert Client-Latenz, Tokens/s und den VRAM-Stand beider GPUs; außerdem wird geprüft, dass ein ungültiger Token abgewiesen wird.

In [ ]:
probe_schema = {
    "type": "object",
    "properties": {"ok": {"type": "boolean"}, "answer": {"type": "string"}},
    "required": ["ok", "answer"],
    "additionalProperties": False,
}
probe_started = time.perf_counter()
probe = client.chat.completions.create(
    messages=[{"role": "user", "content": "Antworte kurz auf Deutsch: Ist dieser Gemma-4-Endpunkt bereit?"}],
    response_format={"type": "json_schema", "json_schema": {"name": "health_probe", "strict": True, "schema": probe_schema}},
    max_tokens=120,
    temperature=0,
)
probe_latency = time.perf_counter() - probe_started
probe_json = json.loads(probe.choices[0].message.content)
assert probe_json["ok"] is True
output_tokens = probe.usage.completion_tokens
print(f"Inference-Latenz (lokal, ohne Tunnel): {probe_latency:.1f} s")
print(f"Ausgabe: {output_tokens} Tokens; grob {output_tokens / probe_latency:.2f} Tokens/s")
if output_tokens / probe_latency < 12:
    print("WARNUNG: Unter 12 Tokens/s – das deutet auf CPU-Betrieb hin. Log und nvidia-smi prüfen.")
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total", "--format=csv"],
    capture_output=True, text=True,
).stdout)

bad_auth = httpx.post(
    f"{BASE_URL}/v1/chat/completions",
    headers={"Authorization": "Bearer invalid-test-token"},
    json={"messages": [{"role": "user", "content": "test"}]},
    timeout=30,
)
assert bad_auth.status_code == 401, f"Auth smoke test fehlgeschlagen: {bad_auth.status_code}"
print("Auth-Fehlerszenario OK: ungültiger Token wird mit HTTP 401 abgewiesen.")
print("Bei Fehlern: Log in /kaggle/working/llama-server.log prüfen; bei Kontext-Überschreitung Eingabe kürzen.")

## 8. Server- und Tunnel-Logs verfolgen

Diese Zelle zeigt fortlaufend die für Leadbrief relevanten Zeilen aus `/kaggle/working/llama-server.log` (Anfrage gestartet, Prompt-Größe, Verarbeitungszeiten, Warnungen und Fehler wie Kontextüberschreitung) und Fehler/Warnungen des Tunnels aus `/kaggle/working/cloudflared.log`. Alle Zeiten sind **UTC**, genau wie in den Browser-Logs der Erweiterung.

**Zuordnung zur Erweiterung:** In der Erweiterung die Seitenleiste mit Rechtsklick → *Untersuchen* öffnen und in der Konsole nach `[Leadbrief` filtern. Dort steht je KI-Aufruf z. B. `#3 Analyse gestartet` und `#3 Analyse fertig nach 74.2 s` mit `serverTimings.prompt_n` / `predicted_n`. Diese Werte entsprechen hier `prompt eval time … / N tokens` und `eval time … / N tokens` derselben Uhrzeit.

- Client-Fehler ohne passende Serverzeile → Anfrage kam nicht an (Tunnel-Link, Token, Netzwerk).
- Serverzeile mit `error` → Server hat abgelehnt (z. B. Kontext zu groß).
- Server fertig, Client meldet Verbindungsabbruch → Tunnel hat die Antwort unterbrochen (Tunnel-Zeilen prüfen).

Die Zelle läuft bis zu 60 Minuten; mit **Interrupt** (■) beenden. Andere Zellen laufen in Kaggle erst danach.

In [ ]:
import datetime as dt

_SERVER_LINE = re.compile(r"^(\d+)\.(\d{2})\.(\d{3})\.\d{3} ([IWED]) (.*)$")
_SERVER_RELEVANT = re.compile(r"processing task|new prompt|prompt processing|eval time|total time|stop processing|exceeds|error|failed", re.I)
_TUNNEL_RELEVANT = re.compile(r" (ERR|WRN) ")


def _read_complete_lines(handle):
    lines = []
    while True:
        position = handle.tell()
        line = handle.readline()
        if not line:
            break
        if not line.endswith("\n"):
            handle.seek(position)
            break
        lines.append(line.rstrip("\n"))
    return lines


def _format_server_line(line):
    match = _SERVER_LINE.match(line)
    if not match:
        return None
    minutes, seconds, millis, level, text = match.groups()
    if level not in "WE" and not _SERVER_RELEVANT.search(text):
        return None
    offset = int(minutes) * 60 + int(seconds) + int(millis) / 1000
    stamp = dt.datetime.fromtimestamp(server_started_at + offset, dt.timezone.utc).strftime("%H:%M:%S")
    return f"{stamp} UTC [server {level}] {text.strip()}"


def follow_logs(minutes=60, show_history=True):
    """Zeigt relevante Server- und Tunnelzeilen fortlaufend an; mit Interrupt beenden."""
    sources = {"server": server_log_path, "tunnel": globals().get("cloudflared_log_path", "/kaggle/working/cloudflared.log")}
    handles = {}
    deadline = time.time() + minutes * 60
    print(f"Verfolge Logs bis {dt.datetime.fromtimestamp(deadline, dt.timezone.utc):%H:%M} UTC …")
    try:
        while time.time() < deadline:
            for name, path in sources.items():
                if name not in handles and os.path.exists(path):
                    handles[name] = open(path, errors="replace")
                    if not show_history:
                        handles[name].seek(0, os.SEEK_END)
                for line in _read_complete_lines(handles[name]) if name in handles else []:
                    shown = _format_server_line(line) if name == "server" else (f"[tunnel] {line}" if _TUNNEL_RELEVANT.search(line) else None)
                    if shown:
                        print(shown, flush=True)
            if server_process.poll() is not None:
                print(f"llama-server ist beendet (Exit-Code {server_process.returncode}). Log: {server_log_path}")
                break
            if globals().get("cloudflared") is not None and cloudflared.poll() is not None:
                print(f"Tunnel ist beendet (Exit-Code {cloudflared.returncode}); Abschnitt 5 erneut ausführen.")
                break
            time.sleep(1)
    except KeyboardInterrupt:
        print("Log-Verfolgung beendet.")
    finally:
        for handle in handles.values():
            handle.close()


follow_logs()